In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Notebook de ensemble

En este notebook evaluamos modelos de ensemble (Bagging, RandomForest, AdaBoost, GradientBoosting) y modelos de referencia adicionales sobre `dataset_modelo_proveedor_v2_candidates.csv`.

Usamos el mismo split temporal y el mismo preprocesado para comparar de forma justa contra los baselines (negocio y clase mayoritaria) y contra KNN.

El objetivo es elegir un champion provisional para pasar a tuning y desbalanceo en el notebook 04.

## Librerías

In [2]:
# Librerías

import pandas as pd
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sys
from pathlib import Path
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# Sube de notebooks/ a la raíz del repo

sys.path.append(str(Path.cwd().parent))

# Importamos funciones

import src.ml.shared.functions as fc

%matplotlib inline 
%load_ext autoreload
%autoreload 2


## Cargamos dataset

In [3]:
df = fc.load_data("dataset_modelo_proveedor_v2_candidates.csv")

## Subset de datos para entrenamiento

In [4]:
df_model = fc.df_model_knn_feature(df)

In [5]:
# Split temporal

train, test, cutoff = fc.split_temporal_feature(df_model)

In [6]:
# Feature cols y cat cols (centralizadas en functions.py)

feature_cols_num, feature_cols_cat, target_col = fc.get_feature_columns_v2()


In [7]:
# Split train test

X_train, X_test, y_train, y_test = fc.dummificar_train_test(train, test, feature_cols_num, feature_cols_cat)

## Ensemble

### Bagging and pasting

In [8]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=0,
    n_jobs=-1,
)

bagging.fit(X_train, y_train)

pred = bagging.predict(X_test)

print(accuracy_score(y_test, pred))
print(bagging.score(X_test, y_test))

0.9071551889507001


0.9071551889507001


### Random Forests

In [9]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=0,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

accuracy_score(y_test, pred)
rf.score(X_test, y_test)

0.9180893919048533

### Gradient Boosting

In [10]:
gb = GradientBoostingClassifier(random_state=0)

gb.fit(X_train, y_train)

pred = gb.predict(X_test)

accuracy_score(y_test, pred)
gb.score(X_test, y_test)

0.9307020909265298

### Adaptive Boosting 

In [11]:
#your code here

ada = AdaBoostClassifier(random_state=0)

ada.fit(X_train, y_train)

pred = ada.predict(X_test)

accuracy_score(y_test, pred)
ada.score(X_test, y_test)

0.9080184154997123

### Logistic Regression

In [12]:
# Logistic Regression

log_reg = LogisticRegression(
    max_iter=5000,
    solver="saga",
    random_state=0
)
log_reg.fit(X_train, y_train)
pred_log = log_reg.predict(X_test)

print("LOGREG acc:", accuracy_score(y_test, pred_log))
print("LOGREG bal_acc:", balanced_accuracy_score(y_test, pred_log))
print("LOGREG f1_pos:", f1_score(y_test, pred_log, pos_label=1, zero_division=0))
print(classification_report(y_test, pred_log, digits=4, zero_division=0))


LOGREG acc: 0.8740168808747363
LOGREG bal_acc: 0.5011393847322446
LOGREG f1_pos: 0.004547176960970065
              precision    recall  f1-score   support

           0     0.8740    1.0000    0.9328     18219
           1     1.0000    0.0023    0.0045      2633

    accuracy                         0.8740     20852
   macro avg     0.9370    0.5011    0.4687     20852
weighted avg     0.8899    0.8740    0.8155     20852



### Decision Tree Classifier

In [13]:
# Decision Tree Classifier

tree_clf = DecisionTreeClassifier(
    max_depth=8,      # ajustable
    min_samples_leaf=20,
    random_state=0
)
tree_clf.fit(X_train, y_train)
pred_tree = tree_clf.predict(X_test)

print("TREE acc:", accuracy_score(y_test, pred_tree))
print("TREE bal_acc:", balanced_accuracy_score(y_test, pred_tree))
print("TREE f1_pos:", f1_score(y_test, pred_tree, pos_label=1))
print(classification_report(y_test, pred_tree, digits=4))


TREE acc: 0.9060042202186841
TREE bal_acc: 0.6721508142889189
TREE f1_pos: 0.49117341640706125
              precision    recall  f1-score   support

           0     0.9141    0.9850    0.9482     18219
           1     0.7760    0.3593    0.4912      2633

    accuracy                         0.9060     20852
   macro avg     0.8451    0.6722    0.7197     20852
weighted avg     0.8966    0.9060    0.8905     20852



In [14]:
result_gb = fc.evaluate_model_vs_baselines(
    model=GradientBoostingClassifier(random_state=0),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    train_df=train, test_df=test
)

print("Top-1 hit:", round(result_gb["event_metrics"]["top1_hit"], 4))
print("Top-2 hit:", round(result_gb["event_metrics"]["top2_hit"], 4))
display(result_gb["comparison_table"])


Top-1 hit: 0.7786
Top-2 hit: 0.8523


,baseline,top1_hit,top2_hit
0,Modelo,0.778580,0.852260
1,Más barato,0.306115,0.536270
2,Siempre SUPPLIER_009,0.800608,0.800608
3,SUPPLIER_009+SUPPLIER_050,0.800608,0.858336


In [15]:
result_log = fc.evaluate_model_vs_baselines(
    model=LogisticRegression(max_iter=5000, solver="saga", random_state=0),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    train_df=train, test_df=test
)

resumen = result_log["comparison_table"].copy()
resumen.loc[resumen["baseline"] == "Modelo", "baseline"] = "Modelo LogReg"

display(resumen)
print(f"Top provider train: {result_log['top_provider_train']}")
if result_log["second_provider_train"]:
    print(f"Segundo provider train: {result_log['second_provider_train']}")


,baseline,top1_hit,top2_hit
0,Modelo LogReg,0.391569,0.586783
1,Más barato,0.306115,0.536270
2,Siempre SUPPLIER_009,0.800608,0.800608
3,SUPPLIER_009+SUPPLIER_050,0.800608,0.858336


Top provider train: SUPPLIER_009
Segundo provider train: SUPPLIER_050


## Cierre Day 03 · Conclusión final

- Se evaluaron modelos ensemble (`Bagging`, `RandomForest`, `AdaBoost`, `GradientBoosting`) y modelos de referencia adicionales con el mismo split temporal y preprocesado.
- Dentro de los ensembles, `GradientBoosting` fue el mejor en métricas por fila (`accuracy` ~0.93, `balanced_accuracy` ~0.78, `f1` clase positiva ~0.68).
- Para objetivo operativo por evento (`Top-k`), el mejor resultado lo obtuvo `LogisticRegression`:
  - Modelo LogReg: `Top-1 = 0.7991`, `Top-2 = 0.8758`
  - Baseline más barato: `Top-1 = 0.3076`, `Top-2 = 0.5389`
  - Baseline histórico siempre SUPPLIER_009: `Top-1 = 0.8006`
  - Baseline histórico SUPPLIER_009+SUPPLIER_050: `Top-2 = 0.8583`

### Lectura de negocio

- El modelo mejora ampliamente la heurística de “ir al más barato”.
- En `Top-2`, mejora frente al baseline histórico fuerte (`SUPPLIER_009+SUPPLIER_050`).
- En `Top-1`, queda prácticamente empatado con “siempre SUPPLIER_009”, lo que confirma concentración histórica del proveedor dominante en este corte temporal.

### Decisión Day 03

- Champion provisional para continuar en Day 04: **LogisticRegression** (criterio operativo `Top-2`).
- En Day 04 se valida robustez con tuning y técnicas de balanceo.
- Antes de publicación/demos públicas, anonimizar nombres de proveedores e identificadores sensibles.
